# P4 — Doublons · Métrique · Coût des erreurs

| | |
|---|---|
| **Séance** | 1 |
| **Nom** | TOSSA Yoann |
| **Porteur #** | P4 |
| **Étapes Assignées** | 1.7 · 1.11 · 1.12 |

---

**Sujet 05 — Santander Satisfaction**  
**Porteur P4** · Enchaînement : 1.7 → 1.11 → 1.12

| Section | Titre | Question clé |
|---|---|---|
| 1.7 | Doublons | Y a-t-il des lignes ou colonnes identiques ? |
| 1.11 | La métrique | Sur quoi juge-t-on le modèle ? |
| 1.12 | Le coût des erreurs | Qu'est-ce qu'une erreur coûte à la banque ? |

In [ ]:
# Dépendances
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
# Localisation des données (même logique que le carnet de démarrage)
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_05_Santander_Satisfaction"

def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / "data", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [p / "data" for p in list(ici.parents)[:4]]  # carnet rangé dans un sous-dossier du dépôt
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    essais = []
    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Données introuvables.\n"
        "Placez train.csv et a_decider.csv dans le même dossier que ce carnet,\n"
        "ou définissez DONNEES_PROJET=/chemin/vers/les/données.\n"
        "Chemins essayés : " + str(essais[:12])
    )

DOSSIER = trouver_donnees()
print("Données trouvées :", DOSSIER)

In [ ]:
# Chargement
train = pd.read_csv(DOSSIER / "train.csv")
print(f"train.csv : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")

---
## 1.7 — Doublons

On cherche deux types de doublons :
- **Lignes dupliquées** : deux clients avec exactement les mêmes valeurs sur toutes les colonnes
- **Colonnes dupliquées** : deux variables qui portent exactement les mêmes valeurs

In [ ]:
# --- 1.7a : Doublons de lignes ---
n_doublons_lignes = train.duplicated().sum()
print(f"Lignes dupliquées : {n_doublons_lignes}")

if n_doublons_lignes > 0:
    print("\nExemple de doublons :")
    masque = train.duplicated(keep=False)
    display(train[masque].head(6))
else:
    print("Aucune ligne dupliquée — le jeu est propre sur ce point.")

In [ ]:
# --- 1.7b : Doublons de colonnes ---
# On compare les colonnes deux à deux (hors TARGET)
colonnes = [c for c in train.columns if c != "TARGET"]
groupes_doublons = {}

deja_vu = {}  # hash -> première colonne rencontrée
for col in colonnes:
    h = hash(train[col].values.tobytes())
    if h in deja_vu:
        ref = deja_vu[h]
        if ref not in groupes_doublons:
            groupes_doublons[ref] = []
        groupes_doublons[ref].append(col)
    else:
        deja_vu[h] = col

if groupes_doublons:
    print(f"{sum(len(v) for v in groupes_doublons.values())} colonne(s) dupliquée(s) trouvée(s) :")
    for ref, copies in groupes_doublons.items():
        print(f"  '{ref}' == {copies}")
else:
    print("Aucune colonne dupliquée.")

In [ ]:
# --- 1.7c : Bilan et décision ---
colonnes_a_supprimer = [col for copies in groupes_doublons.values() for col in copies]

print("=== BILAN 1.7 — Doublons ===")
print(f"  Lignes dupliquées     : {n_doublons_lignes}")
print(f"  Colonnes dupliquées   : {len(colonnes_a_supprimer)}")
if colonnes_a_supprimer:
    print(f"  → Ces colonnes peuvent être retirées sans perte d'information : {colonnes_a_supprimer}")
else:
    print("  → Aucune suppression nécessaire.")

---
## 1.11 — La métrique

**Pourquoi pas l'exactitude (accuracy) ?**

Avec 3,96 % de mécontents, un modèle qui prédit *toujours* « satisfait » obtient :
- Exactitude : **96,04 %** — excellent en apparence
- Mais il n'attrape **aucun** client mécontent

La métrique retenue est l'**AUC-ROC** (aire sous la courbe ROC), qui mesure la capacité du modèle à *ordonner* les clients du plus au moins risqué, indépendamment du seuil.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

CIBLE = "TARGET"
y = train[CIBLE]

# Distribution de la cible
print("Distribution de TARGET :")
print(y.value_counts())
print(f"\nTaux de mécontents : {y.mean()*100:.2f} %")

In [ ]:
# --- Référence : modèle naïf (prédit toujours 0) ---
y_pred_naif = np.zeros(len(y))  # prédit toujours "satisfait"
exactitude_naif = (y == 0).mean()

# AUC d'un score aléatoire
np.random.seed(42)
y_pred_aleatoire = np.random.rand(len(y))
auc_aleatoire = roc_auc_score(y, y_pred_aleatoire)

print(f"Modèle naïf (prédit toujours 0) :")
print(f"  Exactitude : {exactitude_naif*100:.2f} %  ← chiffre trompeur")
print(f"  AUC        : 0.5000              ← pas mieux que le hasard")
print(f"\nModèle aléatoire :")
print(f"  AUC        : {auc_aleatoire:.4f}  ← converge vers 0.5")
print(f"\nPlancher métier (règle de la banque sans modèle) : AUC = 0.6890")

In [ ]:
# --- Visualisation : courbe ROC de référence ---
fig, ax = plt.subplots(figsize=(7, 5))

# Courbe aléatoire
ax.plot([0, 1], [0, 1], 'k--', label='Hasard (AUC = 0.50)')

# Plancher métier (ligne horizontale de référence)
ax.axhline(y=0.6890, color='orange', linestyle=':', label='Plancher métier : AUC = 0.6890')

ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("Courbe ROC — références")
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig("../sorties/roc_reference.png", dpi=100)
plt.show()
print("Figure sauvegardée dans sorties/roc_reference.png")

In [ ]:
# --- Résumé métrique ---
print("=== BILAN 1.11 — La métrique ===")
print("  Métrique choisie : AUC-ROC")
print("  Pourquoi : l'exactitude est trompeuse sur données déséquilibrées (3.96% mécontents)")
print("  L'AUC mesure la capacité à ordonner, indépendamment du seuil de décision")
print(f"  Plancher à battre : AUC = 0.6890 (règle métier de la banque)")
print("  Un modèle utile doit dépasser ce plancher.")

---
## 1.12 — Le coût des erreurs

Deux types d'erreurs existent, avec des coûts **asymétriques** :

| Erreur | Description | Conséquence |
|---|---|---|
| **Faux négatif (FN)** | Client mécontent prédit satisfait | La banque ne rappelle pas — le client part en se plaignant |
| **Faux positif (FP)** | Client satisfait prédit mécontent | Appel inutile — coût humain, temps agent |

On compte en **unités d'appel** : un appel inutile coûte **1**, un mécontent raté coûte **r**.
La banque n'a pas fixé r : on étudie donc plusieurs valeurs, avec **r = 10** comme hypothèse de travail.

**Seuil théorique.** Si le modèle donne une probabilité p bien calibrée, rappeler un client coûte en moyenne (1 − p) × 1,
ne pas le rappeler coûte p × r. On rappelle donc dès que **p > 1 / (1 + r)** : à r = 10, dès 9 % de risque.

**Seuil réel.** Ici on n'a pas encore de modèle : on applique le raisonnement au **score de la règle métier**
(la variable dont l'AUC est la plus proche du plancher, hypothèse de l'audit 1.10 : `var15`),
sur le **même découpage que tout le groupe** (80 / 20 stratifié, graine 42). Le seuil devient alors
« combien de clients rappeler, en partant des plus à risque ».

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

# Hypothèses de coût (à ajuster quand la banque aura fixé r)
COUT_FP = 1    # appel inutile (temps agent)
COUT_FN = 10   # mécontent non rappelé (perte client + plainte) -> r = 10
RATIOS = [2, 5, 10, 25, 50]   # valeurs de r étudiées

# Protocole commun du groupe : 80 / 20 stratifié, graine 42
GRAINE = 42
i_app, i_val = train_test_split(train.index, test_size=0.20, stratify=train[CIBLE], random_state=GRAINE)

# Score de la règle métier : var15, sens (grand = risqué ou l'inverse) choisi sur l'apprentissage
VAR_REGLE = "var15"
sens = 1 if roc_auc_score(train.loc[i_app, CIBLE], train.loc[i_app, VAR_REGLE]) >= 0.5 else -1
y_val = train.loc[i_val, CIBLE].to_numpy()
score_val = sens * train.loc[i_val, VAR_REGLE].to_numpy()

print(f"Règle métier : {VAR_REGLE} ({'grand' if sens > 0 else 'petit'} = plus de risque)")
print(f"AUC sur la validation : {roc_auc_score(y_val, score_val):.4f}  (plancher de l'énoncé : 0.6890)")
print(f"Validation : {len(y_val):,} clients dont {int(y_val.sum())} mécontents ({y_val.mean()*100:.2f} %)")

In [ ]:
# --- Coût selon le nombre de clients rappelés ---
# On classe les clients du plus au moins risqué, puis on rappelle les k premiers (k = 0 ... n)
ordre = np.argsort(-score_val, kind="stable")
k = np.arange(len(y_val) + 1)
trouves = np.r_[0, np.cumsum(y_val[ordre])]   # mécontents trouvés en rappelant les k premiers
n_mecontents = int(y_val.sum())

def cout_total(r):
    # appels inutiles x 1  +  mécontents ratés x r
    return (k - trouves) * COUT_FP + (n_mecontents - trouves) * r

lignes = []
for r in RATIOS:
    c = cout_total(r)
    kb = int(np.argmin(c))
    lignes.append({
        "r (raté / appel inutile)": r,
        "seuil théorique 1/(1+r)": round(1 / (1 + r), 3),
        "appels optimaux": kb,
        "part rappelée": f"{kb / len(y_val) * 100:.1f} %",
        "trouvés": int(trouves[kb]),
        "ratés": n_mecontents - int(trouves[kb]),
        "coût optimal": int(c[kb]),
        "coût sans appel": int(c[0]),
        "coût tout appeler": int(c[-1]),
    })
tableau_couts = pd.DataFrame(lignes)
display(tableau_couts)

**Lecture.** Avec la règle métier, tant que r ≤ 10 la décision la moins chère est de **ne rappeler personne** :
le score est trop peu précis, chaque mécontent trouvé coûte plus d'appels inutiles qu'il ne rapporte.
Il faut que rater un mécontent coûte environ 25 fois un appel pour que rappeler devienne rentable.
C'est exactement ce que le modèle devra améliorer : un meilleur classement rend les appels rentables à des r plus bas.

In [ ]:
# --- Visualisation : coût et rappel selon la part de clients rappelés ---
part = k / len(y_val)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for r in RATIOS:
    c = cout_total(r)
    axes[0].plot(part * 100, c / c[0], label=f"r = {r}")
    axes[0].scatter(part[np.argmin(c)] * 100, c.min() / c[0], s=25)
axes[0].axhline(1, color="grey", linestyle=":", linewidth=1)
axes[0].set_xlabel("Part des clients rappelés (%)")
axes[0].set_ylabel("Coût relatif (1 = ne rappeler personne)")
axes[0].set_title("Coût total selon le nombre d'appels — règle métier")
axes[0].legend()

axes[1].plot(part * 100, trouves / n_mecontents * 100, color="green", label="Règle métier (var15)")
axes[1].plot([0, 100], [0, 100], "k--", label="Hasard")
axes[1].set_xlabel("Part des clients rappelés (%)")
axes[1].set_ylabel("Mécontents trouvés (%)")
axes[1].set_title("Rappel selon le nombre d'appels")
axes[1].legend()

plt.tight_layout()
plt.savefig("../sorties/cout_erreurs.png", dpi=100)
plt.show()
print("Figure sauvegardée dans sorties/cout_erreurs.png")

In [ ]:
# --- Cas d'un budget d'appels fixé ---
# Si la banque ne peut passer qu'un nombre limité d'appels, c'est le budget qui fixe le seuil :
# on rappelle les plus à risque jusqu'à épuisement.
lignes = []
for budget in [0.05, 0.10, 0.20]:
    kb = int(budget * len(y_val))
    lignes.append({
        "budget": f"{int(budget * 100)} % des clients",
        "appels": kb,
        "mécontents trouvés": int(trouves[kb]),
        "rappel": f"{trouves[kb] / n_mecontents * 100:.1f} %",
        "précision (appels utiles)": f"{trouves[kb] / kb * 100:.1f} %",
        f"coût (r = {COUT_FN})": int(cout_total(COUT_FN)[kb]),
    })
display(pd.DataFrame(lignes))

In [ ]:
# --- Matrice de confusion au seuil optimal pour r = COUT_FN ---
c = cout_total(COUT_FN)
k_opt = int(np.argmin(c))
pred_optimal = np.zeros(len(y_val), dtype=int)
pred_optimal[ordre[:k_opt]] = 1
tn, fp, fn, tp = confusion_matrix(y_val, pred_optimal, labels=[0, 1]).ravel()

print(f"=== Matrice de confusion au seuil optimal (r = {COUT_FN}, {k_opt} appels) ===")
print(f"  Vrais Positifs  (TP) : {tp:,}  — mécontents bien détectés")
print(f"  Faux Positifs   (FP) : {fp:,}  — appels inutiles")
print(f"  Faux Négatifs   (FN) : {fn:,}  — mécontents ratés  ← le plus coûteux")
print(f"  Vrais Négatifs  (TN) : {tn:,}  — satisfaits bien ignorés")
print(f"\n  Coût total : {fn}×{COUT_FN} + {fp}×{COUT_FP} = {fn*COUT_FN + fp*COUT_FP:,} unités")

In [ ]:
# --- BILAN FINAL P4 ---
r_rentable = next((l["r (raté / appel inutile)"] for l in tableau_couts.to_dict("records") if l["appels optimaux"] > 0), None)

print("\n=== BILAN P4 — Doublons · Métrique · Coût des erreurs ===")
print()
print("1.7  Doublons")
print(f"     Lignes dupliquées  : {n_doublons_lignes}")
print(f"     Colonnes dupliquées: {len(colonnes_a_supprimer)}")
print()
print("1.11 Métrique : AUC-ROC")
print("     L'exactitude est trompeuse sur données déséquilibrées.")
print("     Plancher à battre : AUC = 0.6890")
print()
print("1.12 Coût des erreurs")
print(f"     Unité : appel inutile = {COUT_FP}, mécontent raté = r (hypothèse de travail r = {COUT_FN})")
print(f"     Seuil théorique (probabilités calibrées) : p > 1/(1+r) = {1/(1+COUT_FN):.3f}")
print(f"     Règle métier ({VAR_REGLE}) à r = {COUT_FN} : {k_opt} appels optimaux, coût {int(c[k_opt]):,}")
print(f"     Rappeler devient rentable avec la règle métier à partir de r = {r_rentable}")
print("     → Le seuil se fixe avec r (ou le budget d'appels) : à demander à la banque.")
print("     → Un meilleur modèle doit rendre les appels rentables pour des r plus bas.")